# Configuration

### Importing Modules 
To import modules ad the same level in the file tree, we need to set the module path.

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    

### Cadquery Viewer Setup 

In [2]:
from cadquery import Workplane
from cadq_server import cq_helper

import cadquery as cq
from jupyter_cadquery import set_sidecar
from jupyter_cadquery import show, show_object, open_viewer, set_defaults

cv = open_viewer('CadQuery')

top_position = (192+90, 220, 300)
top_target = (192+90, 220, 0)
top_zoom = 1
    
set_defaults(
    #viewer=None,
    cad_width=800, 
    height=800, 
    tree_width=250,
    axes=False,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = top_position,
    target = top_target,
    zoom = top_zoom,
    
    #target=(0,0,200),
    #tools=False,
    glass=True,
    show_parent=False,
    #timeit=False
)

Overwriting auto display for cadquery Workplane and Shape


# Wing Construction

### Wing Configuration

In [3]:
from Airplane.aircraft_topology.WingConfiguration import WingConfiguration

airfoil = "../components/airfoils/naca2415.dat"
wing_config = WingConfiguration(
                  root_airfoil=airfoil,
                  nose_pnt=(192.113, 0, -44.5),
                  root_chord=183,
                  root_dihedral=3.7,
                  root_incidence=3,
                  length=410,
                  sweep=20,
                  tip_chord=183,
                  tip_dihedral=25,
                  tip_incidence=0)
wing_config.add_segment(length=100,
                       sweep=20,
                       tip_chord=183-20,
                       tip_dihedral=15,
                       tip_incidence=0)
#wing_config.add_segment(length=5,
#                       sweep=5,
#                       tip_chord=183-20-5,
#                       tip_dihedral=15,
#                       tip_incidence=0)
#wing_config.add_segment(length=5,
#                       sweep=5,
#                       tip_chord=183-20-10,
#                       tip_dihedral=15,
#                       tip_incidence=0)
#wing_config.add_segment(length=5,
#                       sweep=5,
#                       tip_chord=183-20-15,
#                       tip_dihedral=15,
#                       tip_incidence=0)
#wing_config.add_segment(length=50,
#                       sweep=45+50,
#                       tip_chord=183-20-20-40-50,
#                       tip_dihedral=0,
#                       tip_incidence=0,
#                       tip_airfoil="../components/airfoils/nacam2.dat")
wing_configuration = {"main_wing": wing_config}


In [4]:
printer_wall_thickness:float=0.42
spare_support_geometry_is_round:bool=False
spare_support_dimension_width:float=6
spare_support_dimension_height:float=6
spare_perpendicular:bool= False
spare_position_factor:float=1/3 # value betweein (0,1) as fraction of the chord
leading_edge_offset:float=10
trailing_edge_offset:float=30
minimum_rib_angle:float=45

In [5]:
set_defaults(
    axes=True,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = (190,200,300),
    target = (190,200,0),
)

## Vase Mode Rib Construction

Vase mode is a mode where the 3D printer does not need to change a layer, as layer changes are done contiously in spiral motion. To print like that, we need to construct a wing where everything can be printed in one motion, in each layer.

### Construction Workplane
First of all we need a workplane that is defined by the wing chord (including the angle of incidence) and the wing's dehedral.

In [6]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def _wing_workplane(wing_config:WingConfiguration, segment:int = 0) -> Workplane:
    """
    Creating a workplan where the 0-point is located at the wing's nose point
    and the workplane is going through the wing.
    
    Remark: an incident angle at the wing_tip cannot be covered with this 
    workplane.
    """
    origin = np.array((0,0,0,1)) #wing_config.nose_pnt
    xdir = [1,0,0,1]
    normal = [0,0,1,1]
    seg = 0
    
    all_trans = [
            [1, 0, 0, 0],
            [0, 1, 0, 0],
            [0, 0, 1, 0],
            [0, 0, 0, 1]]
    
    for seg in reversed(range(segment)):
        t_sweep_length = [
            [1, 0, 0, wing_config.segments[seg].sweep],
            [0, 1, 0, wing_config.segments[seg].length],
            [0, 0, 1, 0],
            [0, 0, 0, 1]]
        
        r_tip_dihedral = R.from_euler('x', wing_config.segments[seg].tip_dihedral, degrees=True)
        r_tip_dihedral = r_tip_dihedral.as_matrix().copy()
        r_tip_dihedral = np.hstack((r_tip_dihedral,[[0],[0],[0]]))
        r_tip_dihedral = np.vstack((r_tip_dihedral,[0,0,0,1]))
        
        r_tip_incidence = R.from_euler('y', wing_config.segments[seg].tip_incidence, degrees=True)
        r_tip_incidence = r_tip_incidence.as_matrix().copy()
        r_tip_incidence = np.hstack((r_tip_incidence,[[0],[0],[0]]))
        r_tip_incidence = np.vstack((r_tip_incidence,[0,0,0,1]))
        
        all_trans = np.matmul( r_tip_dihedral, all_trans)
        all_trans = np.matmul( r_tip_incidence, all_trans)
        all_trans = np.matmul( t_sweep_length, all_trans)
    
    r_root_incidence = R.from_euler('y', wing_config.segments[seg].root_incidence, degrees=True)
    r_root_incidence = r_root_incidence.as_matrix().copy()
    r_root_incidence = np.hstack((r_root_incidence,[[0],[0],[0]]))
    r_root_incidence = np.vstack((r_root_incidence,[0,0,0,1]))

    r_root_dihedral = R.from_euler('x', wing_config.segments[seg].root_dihedral, degrees=True)
    r_root_dihedral = r_root_dihedral.as_matrix().copy()
    r_root_dihedral = np.hstack((r_root_dihedral,[[0],[0],[0]]))
    r_root_dihedral = np.vstack((r_root_dihedral,[0,0,0,1]))

    all_trans = np.matmul(r_root_dihedral, all_trans)
    all_trans = np.matmul(r_root_incidence, all_trans)
    
    normal = all_trans.transpose()[2]
    origin = all_trans.transpose()[3]
    xdir = all_trans.transpose()[0]
    
    plane = cq.Plane(origin=origin.tolist()[:3], xDir=xdir.tolist()[:3], normal=normal.tolist()[:3])

    wp_plane = (cq.Workplane(inPlane=plane, origin=origin))
    
    return wp_plane

#VaseModeRibCutoutCreator.wing_workplane = _wing_workplane

In [7]:
set_defaults(
    axes=True,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = (-300,250,0),
    target = (0,250,0),
)

In [8]:
segment = 0
wing_wp = _wing_workplane(wing_configuration['main_wing'],segment)
box = wing_wp.box(wing_configuration['main_wing'].segments[segment].root_chord,
            wing_configuration['main_wing'].segments[segment].length,
            2, centered=False)


In [9]:
try:
    segment = segment +1
    wing_wp = _wing_workplane(wing_configuration['main_wing'],segment)
    box = wing_wp.box(wing_configuration['main_wing'].segments[segment].root_chord,
            wing_configuration['main_wing'].segments[segment].length,
            2, centered=False).add(box)
except IndexError:
    print('no segment')

In [10]:
try:
    segment = segment +1
    wing_wp = _wing_workplane(wing_configuration['main_wing'],segment)
    box = wing_wp.box(wing_configuration['main_wing'].segments[segment].root_chord,
            wing_configuration['main_wing'].segments[segment].length,
            2, centered=False).add(box)
except IndexError:
    print('no segment')

no segment


In [11]:
try:
    segment = segment +1
    wing_wp = _wing_workplane(wing_configuration['main_wing'],segment)
    box = wing_wp.box(wing_configuration['main_wing'].segments[segment].root_chord,
            wing_configuration['main_wing'].segments[segment].length,
            2, centered=False).add(box)
except IndexError:
    print('no segment')

no segment


In [12]:
try:
    segment = segment +1
    wing_wp = _wing_workplane(wing_configuration['main_wing'],segment)
    box = wing_wp.box(wing_configuration['main_wing'].segments[segment].root_chord,
            wing_configuration['main_wing'].segments[segment].length,
            2, centered=False).add(box)
except IndexError:
    print('no segment')

no segment


In [13]:
try:
    segment = segment +1
    wing_wp = _wing_workplane(wing_configuration['main_wing'],segment)
    box = wing_wp.box(wing_configuration['main_wing'].segments[segment].root_chord,
            wing_configuration['main_wing'].segments[segment].length,
            2, centered=False).add(box)
except IndexError:
    print('no segment')

no segment


In [14]:
cv = show(box,
    axes=True,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = (-190,200,0),
    target = (190,200,0),
)

Camera control changed, so camera was resetted


## Wing Trapezoid Calculation

We need to calculate the wing's trapezoid outlines as boundaries.

In [15]:
set_defaults(
    axes=True,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = (100,200,300),
    target = (100,200,0),
    zoom=0.5
)

### _calc_edge_start

Calclating the seamless start of the next segment.

In [16]:
from typing import Tuple, cast as tcast

def _calc_edge_start(sketch: cq.Sketch, id_s: str, spare_nose_tip, tip_nose) -> Tuple[float, float, bool]:

    try:
        p_le = sketch.segmentToEdge('spare_nose',180,'rib_nl'+id_s,'helper')._tags['helper'][0].endPoint()
        sketch.select('helper').delete()
        p_te = sketch.segmentToEdge('spare_tail',180,'rib_tl'+id_s,'helper')._tags['helper'][0].endPoint()
        sketch.select('helper').delete()
        lower_part = True
        if ( (spare_nose_tip[0] - p_le.x) > 0 and abs(p_le.y - tip_nose[1]) < 1e-5):
            pass
        else:
            p_le = sketch.segmentToEdge('spare_nose',180,'rib_nr'+id_s,'helper')._tags['helper'][0].endPoint()
            sketch.select('helper').delete()
            p_te = sketch.segmentToEdge('spare_tail',180,'rib_tr'+id_s,'helper')._tags['helper'][0].endPoint()
            sketch.select('helper').delete()
            lower_part = False
    except:
        p_le = sketch.segmentToEdge('spare_nose',180,'rib_nr'+id_s,'helper')._tags['helper'][0].endPoint()
        sketch.select('helper').delete()
        p_te = sketch.segmentToEdge('spare_tail',180,'rib_tr'+id_s,'helper')._tags['helper'][0].endPoint()
        sketch.select('helper').delete()
        lower_part = False
        
    leading_edge_start = p_le.x - tip_nose[0]
    trailing_edge_start = p_te.x - tip_nose[0]
    return (leading_edge_start, trailing_edge_start, lower_part)

### _rib_cutout

In [17]:
from typing import Tuple, cast as tcast
from cadquery.occ_impl.shapes import Edge
from cadquery.occ_impl.geom import Vector, Location
from scipy.spatial.transform import Rotation as R
from cq_plugins.segmentToEdge import segmentToEdge

def _rib_cutout(
        segment: int,
        wing_config: WingConfiguration,
        printer_wall_thickness:float,
        spare_support_dimension_width:float,
        spare_support_dimension_height:float,
        leading_edge_offset:float,
        trailing_edge_offset:float,
        leading_edge_start: float = None,
        trailing_edge_start: float = None,
        start_upper_part: bool = False, # construction starts with the upper part of the hourglas structure
        minimum_rib_angle:float=45,
        spare_perpendicular:bool= False,
        spare_position_factor:float=1./3., 
        ) -> cq.Sketch:
    '''
    Constructs a set of hourglass like structures in between the nose and the tail part of the wing.
    
    '''
    
    
    if leading_edge_start == None:
        leading_edge_start = leading_edge_offset
    if trailing_edge_start == None:
        trailing_edge_start = wing_config.segments[segment].root_chord - trailing_edge_offset

    root_nose = np.asarray((.0, .0, .0))
    root_nose_offset = root_nose + np.asarray((leading_edge_offset, .0, .0)) 
    root_nose_start = np.asarray((leading_edge_start, .0, .0))

    root_tail = np.asarray((1.,.0, .0)) * wing_config.segments[segment].root_chord
    root_tail_offset = root_tail - np.asarray((trailing_edge_offset,.0, .0))
    root_tail_start = np.asarray((trailing_edge_start, .0, .0))

    tip_nose = np.asarray((wing_config.segments[segment].sweep, wing_config.segments[segment].length, 0.))
    tip_nose_offset = tip_nose + np.asarray((1.0, .0, .0)) * leading_edge_offset

    tip_tail = tip_nose + np.asarray((1.,.0, .0)) * wing_config.segments[segment].tip_chord
    tip_tail_offset = tip_tail - np.asarray((1.,.0, .0)) * trailing_edge_offset

    # Calculating the spare nose and tail positions
    spare_nose_root = (  np.asarray((1.,0.,0.)) * wing_config.segments[segment].root_chord 
                       * spare_position_factor 
                       - np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2  
                       - np.asarray((1.,0.,0.)) *  printer_wall_thickness )
    spare_tail_root = ( np.asarray((1.,0.,0.)) * wing_config.segments[segment].root_chord 
                       * spare_position_factor 
                       + np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2  
                       + np.asarray((1.,0.,0.)) *  printer_wall_thickness )

    if not spare_perpendicular:
        spare_nose_tip = ( tip_nose + np.asarray((1.,0.,0.)) * wing_config.segments[segment].tip_chord 
                          * spare_position_factor 
                          - np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2 
                          - np.asarray((1.,0.,0.)) *  printer_wall_thickness )
        spare_tail_tip = ( tip_nose + np.asarray((1.,0.,0.)) * wing_config.segments[segment].tip_chord 
                          * spare_position_factor 
                          + np.asarray((1.,0.,0.)) *  spare_support_dimension_width/2 
                          + np.asarray((1.,0.,0.)) *  printer_wall_thickness )
    else:
        spare_nose_tip = ( spare_nose_root
                          + np.asarray((0.,1.,0.)) *  wing_config.segments[segment].length )
        spare_tail_tip = ( spare_tail_root
                          + np.asarray((0.,1.,0.)) *  wing_config.segments[segment].length )
        
    # Drawing the offset outlines in the sketch.
    const_lines = (
        cq.Sketch()              
        .segment(Vector(tuple(root_nose_offset)), 
                Vector(tuple(tip_nose_offset)),'nose_os', forConstruction=True)
        .segment(Vector(tuple(root_tail_offset)), 
                Vector(tuple(tip_tail_offset)),'tail_os', forConstruction=True)
        .segment(Vector(tuple(spare_tail_root)), 
                Vector(tuple(spare_tail_tip)),'spare_tail', forConstruction=True)
        .segment(Vector(tuple(spare_nose_root)), 
                Vector(tuple(spare_nose_tip)),'spare_nose', forConstruction=True)
    )

    # Constructing the first row of ribs...
    if not start_upper_part:
        const_lines = (
            const_lines
            .segmentToEdge(Vector(tuple(root_tail_start)),
                        180.-minimum_rib_angle, 'spare_tail', 'rib_tl') # rib: tail left  \
            .segmentToEdge(minimum_rib_angle, 'tail_os', 'rib_tr')     # rib: tail right /
            .segmentToEdge(180.,'nose_os', 'help_top', forConstruction=False)
            .segmentToEdge('rib_tl', 180.,'spare_nose', 'help_middle', forConstruction=False)
            .segment(Vector(tuple(root_nose_start)),'rib_nl')          # rib: nose left  /
            .segmentToEdge('help_middle', 1, 'help_top', 1., 'rib_nr')        # rib: nose right \
            .segment(Vector(tuple(root_tail_start)),Vector(tuple(root_tail_start))
                     -Vector((0,wing_config.segments[segment].length*0.1,0)),'nose_ext')
            .segment(Vector(tuple(root_nose_start)),Vector(tuple(root_nose_start))
                    -Vector((0,wing_config.segments[segment].length*0.1,0)),'tail_ext')
            #.segment(Vector(tuple(root_tail_start)),Vector(tuple(root_nose_start)),'root')
            .segmentToEdge('nose_ext',1.,'tail_ext',1.,'root')
            )
    else:
        const_lines = (
            const_lines
            .segmentToEdge(Vector(tuple(root_tail_start)), minimum_rib_angle, 'tail_os', 'rib_tr')     # rib: tail right (upper) /
            .segmentToEdge(180.,'nose_os', 'help_top', forConstruction=False)
            .segmentToEdge('help_top', 1., Vector(tuple(root_nose_start)), 'rib_nr') # rib: nose right (upper) \
            .segment(Vector(tuple(root_tail_start)),Vector(tuple(root_tail_start))
                     -Vector((0,wing_config.segments[segment].length*0.1,0)),'nose_ext')
            .segment(Vector(tuple(root_nose_start)),Vector(tuple(root_nose_start))
                    -Vector((0,wing_config.segments[segment].length*0.1,0)),'tail_ext')
            #.segment(Vector(tuple(root_tail_start)),Vector(tuple(root_nose_start)),'root')
            .segmentToEdge('nose_ext',1.,'tail_ext',1.,'root')
            )
    show(const_lines)

    # health check
    # 'rib_nr' should not end left of 'rib_tr'
    if (tcast(Edge, const_lines._tags['rib_nr'][0]).endPoint().x >
        tcast(Edge, const_lines._tags['rib_tr'][0]).endPoint().x):
        start_p = tcast(Edge, const_lines._tags['rib_nr'][0]).startPoint()
        const_lines = (
            const_lines
            .select('rib_nr').delete()
            .segmentToEdge(start_p, 180-minimum_rib_angle, 'nose_os', 'rib_nr') # rib: nose right (upper) \
            .select('help_top').delete()
            .segmentToEdge('rib_tr', 1.,  'rib_nr', 1.,'help_top') # rib: nose right (upper) \
        )
        
    # Constructing as many ribs as do fit in the wing.
    id_s = ''

    while (tcast(Edge, const_lines._tags['help_top'+id_s][0]).startPoint().y 
           < wing_config.segments[segment].length):
        const_lines = (
            const_lines
            .segmentToEdge('rib_tr'+id_s, 180-minimum_rib_angle, 'spare_tail', 'rib_tl_'+id_s) 
            .segmentToEdge(minimum_rib_angle, 'tail_os', 'rib_tr_'+id_s)  
            .segmentToEdge(180,'nose_os', 'help_top_'+id_s, forConstruction=False)
            .segmentToEdge('rib_tl_'+id_s, 180,'spare_nose', 'help_middle_'+id_s, forConstruction=False)
            .segmentToEdge('help_top'+id_s, 1, 'help_middle_'+id_s, 1,'rib_nl_'+id_s)
            .segmentToEdge('help_middle_'+id_s, 1, 'help_top_'+id_s, 1., 'rib_nr_'+id_s)
            .select('help_top'+id_s).delete()
            )
        if not start_upper_part:
            const_lines.select('help_middle'+id_s).delete()

        # health check
        # 'rib_nr' should not end left of 'rib_tr'
        if (tcast(Edge, const_lines._tags['rib_nr_'+id_s][0]).endPoint().x >
            tcast(Edge, const_lines._tags['rib_tr_'+id_s][0]).endPoint().x):
            start_p = tcast(Edge, const_lines._tags['rib_nr'+id_s][0]).startPoint()
            const_lines = (
                const_lines
                .select('rib_nr_'+id_s).delete()
                .segmentToEdge(start_p, 180-minimum_rib_angle, 'nose_os', 'rib_nr_'+id_s) # rib: nose right (upper) \
                .select('help_top').delete()
                .segmentToEdge('rib_tr', 1.,  'rib_nr', 1.,'help_top') # rib: nose right (upper) \
            )
        id_s = id_s + '_'
    show(const_lines)

    # Removing all constrution lines...
    if not start_upper_part:
        const_lines.select('help_middle'+id_s).delete()
    const_lines.select('nose_os').delete()
    const_lines.select('spare_nose').delete()
    const_lines.select('spare_tail').delete()
    const_lines.select('tail_os').delete()
    show(const_lines)
    return const_lines, *_calc_edge_start(const_lines, id_s, spare_nose_tip, tip_nose)

#### Test a segment

In [18]:
leading_edge_start=None
trailing_edge_start=None
lower_part=True
selected_segment = -1

In [19]:
selected_segment = selected_segment +1
cutout_face, leading_edge_start, trailing_edge_start, lower_part = _rib_cutout(
        segment=selected_segment,
        wing_config=wing_config,
        printer_wall_thickness=printer_wall_thickness,
        spare_support_dimension_width=spare_support_dimension_width,
        spare_support_dimension_height=spare_support_dimension_height,
        leading_edge_offset=leading_edge_offset,
        trailing_edge_offset=trailing_edge_offset,
        leading_edge_start=leading_edge_start,
        trailing_edge_start=trailing_edge_start,
        start_upper_part= not lower_part,
        minimum_rib_angle=minimum_rib_angle,
        spare_perpendicular=spare_perpendicular,
        spare_position_factor=spare_position_factor)
cutout_face
print(f"lower_part:{lower_part}") 

lower_part:True


Assembling the final face.

In [20]:
cutout_face.assemble()

Placing the face in the wing's workplane and extruding the face and intersecting it with the wing.

In [21]:
from Airplane.creator.WingLoftCreator import WingLoftCreator

wing_wp = _wing_workplane(wing_configuration['main_wing'],0)

wlc = WingLoftCreator(creator_id="wing",
                 wing_index="main_wing",
                 wing_config=wing_configuration)
wlc_wing: Workplane = wlc._create_shape([],[])['wing']
wlc_wing=wlc_wing.translate(-Vector(wing_config.nose_pnt))
wlc_wing

In [22]:
wlc_of = WingLoftCreator(creator_id="wing.offset",
                 wing_index="main_wing",
                 offset=printer_wall_thickness*2,
                 wing_config=wing_configuration)
wlc_wing_of: Workplane = wlc_of._create_shape([],[])['wing.offset']
#wlc_wing_of=wlc_wing_of.translate(-Vector(wing_config.nose_pnt))
#wlc_wing_of

In [23]:
wing_wp.placeSketch(cutout_face).add(wlc_wing).cutThruAll() #.extrude(until=100, taper=0, both=True).intersect(wlc_wing)

In [24]:
print(f"{leading_edge_start}:{trailing_edge_start}")

40.890873786407695:95.49025641025648


## VaseModeRibCutoutCreator

### VaseModeRibCutoutCreator

In [25]:
import logging
from typing import Union, Literal

from cadquery import Workplane

from Airplane.AbstractShapeCreator import AbstractShapeCreator
from Airplane.aircraft_topology.WingConfiguration import WingConfiguration
from cq_plugins.wing.wing_segment import wing_segment
from cq_plugins.wing.wing_root_segment import wing_root_segment
from cq_plugins.fix_shape.fix_shape import fix_shape

class VaseModeRibCutoutCreator(AbstractShapeCreator):
    """
    Create a cutout shape that should be intersected with the hull shape.
    The shape :
               | \  ||     / | trailing
    leading    |  \ ||   /   | edge
    edge       |   \||/      |
               |   /||\      |
               |  / ||   \   |
               | /  ||     \ |
               offset        offset
    """

    def __init__(self, creator_id: str,
                 wing_index: Union[str, int],
                 printer_wall_thickness: float,
                 spare_support_geometry_is_round: bool,
                 spare_support_dimension_width: float,
                 spare_support_dimension_height: float,
                 leading_edge_offset: float,
                 trailing_edge_offset: float,
                 spare_position_factor: float = 1. / 3.,
                 minimum_rib_angle: float = 45,
                 spare_perpendicular: bool = False,
                 invert_cutout: bool = False,
                 taper_cutout: float = 0.,
                 wing_config: dict[int, WingConfiguration] = None,
                 wing_side: Literal["LEFT", "RIGHT", "BOTH"] = "RIGHT",
                 loglevel: int =logging.INFO):
        """
        parameters:
        printer_wall_thickness - printer settings wall thickness
        spare_support_geometry_is_round -- default true
        spare_support_dimension_x -- diameter if round is true
        spare_support_dimension_z -- ignored if round
        leading_edge_offset --
        trailing_edge_offset --
        minimum_rib_angle -- important for printability (should be > 45°)
        """
        self.printer_wall_thickness: float = printer_wall_thickness
        self.spare_support_geometry_is_round: bool = spare_support_geometry_is_round
        self.spare_support_dimension_width: float = spare_support_dimension_width
        self.spare_support_dimension_height: float = spare_support_dimension_height
        self.spare_perpendicular: bool = spare_perpendicular
        self.spare_position_factor: float = spare_position_factor
        self.leading_edge_offset: float = leading_edge_offset
        self.trailing_edge_offset: float = trailing_edge_offset
        self.minimum_rib_angle: float = minimum_rib_angle
        self.invert_cutout: bool = invert_cutout
        self.taper_cutout: float = taper_cutout
        self.wing_side: Literal["LEFT", "RIGHT", "BOTH"] = wing_side
        self.wing_index: Union[str, int] = wing_index
        self._wing_config: dict[int, WingConfiguration] = wing_config

        super().__init__(creator_id, shapes_of_interest_keys=[], loglevel=loglevel)

    def _create_shape(self, shapes_of_interest: dict[str, Workplane],
                      input_shapes: dict[str, Workplane],
                      **kwargs) -> dict[str, Workplane]:
        logging.info(f"wing rib cutout from configuration --> '{self.identifier}'")
        wall_factor = 1

        wing_config: WingConfiguration = self._wing_config[self.wing_index]
        right_wing: Workplane = (
            Workplane('XZ')
            .wing_root_segment(
                root_airfoil=wing_config.segments[0].root_airfoil,
                root_chord=wing_config.segments[0].root_chord,
                root_dihedral=wing_config.segments[0].root_dihedral,
                root_incidence=wing_config.segments[0].root_incidence,
                length=wing_config.segments[0].length,
                sweep=wing_config.segments[0].sweep,
                tip_chord=wing_config.segments[0].tip_chord,
                tip_dihedral=wing_config.segments[0].tip_dihedral,
                tip_incidence=wing_config.segments[0].tip_incidence,
                tip_airfoil=wing_config.segments[0].tip_airfoil,
                offset=self.printer_wall_thickness * wall_factor))

        cutout_face, leading_edge_start, trailing_edge_start, lower_part = _rib_cutout(
            segment=0,
            wing_config=wing_config,
            printer_wall_thickness=self.printer_wall_thickness,
            spare_support_dimension_width=self.spare_support_dimension_width,
            spare_support_dimension_height=self.spare_support_dimension_height,
            leading_edge_offset=self.leading_edge_offset,
            trailing_edge_offset=self.trailing_edge_offset,
            minimum_rib_angle=self.minimum_rib_angle,
            spare_perpendicular=self.spare_perpendicular,
            spare_position_factor=self.spare_position_factor)
        # Assembling the face.
        cutout_face.assemble()

        if self.invert_cutout:
            right_wing_cutout = (_wing_workplane(wing_config, segment=0)
                                 .placeSketch(cutout_face)
                                 .extrude(until=100, taper=self.taper_cutout, both=True)
                                 .intersect(right_wing)
                                 )
        else:
            right_wing_cutout = (_wing_workplane(wing_config, segment=0)
                                 .placeSketch(cutout_face)
                                 .add(right_wing)
                                 .cutThruAll(taper=self.taper_cutout)
                                 )

        segment = 0
        current: Workplane = right_wing
        for segment_config in wing_config.segments[1:]:
            segment = segment + 1
            current = current.wing_segment(
                length=segment_config.length,
                sweep=segment_config.sweep,
                tip_chord=segment_config.tip_chord,
                tip_dihedral=segment_config.tip_dihedral,
                tip_incidence=segment_config.tip_incidence,
                tip_airfoil=segment_config.tip_airfoil,
                offset=self.printer_wall_thickness * wall_factor)
            #wing_seg_solid: Solid = wing_seg.vals()[-1]
            #wing_seg_solid.move(wing_seg.vals()[-3])
            right_wing.add(current)

            cutout_face, leading_edge_start, trailing_edge_start, lower_part = _rib_cutout(
                segment=segment,
                wing_config=wing_config,
                printer_wall_thickness=self.printer_wall_thickness,
                spare_support_dimension_width=self.spare_support_dimension_width,
                spare_support_dimension_height=self.spare_support_dimension_height,
                leading_edge_offset=self.leading_edge_offset,
                trailing_edge_offset=self.trailing_edge_offset,
                leading_edge_start=leading_edge_start,
                trailing_edge_start=trailing_edge_start,
                minimum_rib_angle=self.minimum_rib_angle,
                spare_perpendicular=self.spare_perpendicular,
                spare_position_factor=self.spare_position_factor)

            # Assembling the face.
            cutout_face.assemble()
            #show(cutout_face)

            try:
                if self.invert_cutout:
                    right_wing_cutout.add(
                        _wing_workplane(wing_config, segment=segment)
                        .placeSketch(cutout_face)
                        .extrude(until=100, taper=self.taper_cutout, both=True)
                        .intersect(current)
                    )
                else:
                    right_wing_cutout.add(
                        _wing_workplane(wing_config, segment=segment)
                        .placeSketch(cutout_face)
                        .add(current)
                        .cutThruAll(taper=self.taper_cutout)
                    )
                print(f"seg: {segment}")
                #show(right_wing_cutout)
                pass
            except:
                logging.warning(f"could not create segment: {segment}!")
            pass

        if self.wing_side == "LEFT":
            right_wing = right_wing.mirror("XZ")
            right_wing_cutout = right_wing_cutout.mirror("XZ")
        elif self.wing_side == "BOTH":
            left_wing = right_wing.mirror("XZ")
            right_wing = right_wing.union(left_wing)
            left_wing_cutout = right_wing_cutout.mirror("XZ")
            right_wing_cutout = right_wing_cutout.union(left_wing_cutout)

        right_wing = right_wing.fix_shape()
        right_wing = right_wing.translate(wing_config.nose_pnt).display(name=f"{self.identifier}",
                                                                        severity=logging.DEBUG)
        right_wing_cutout = right_wing_cutout.fix_shape()
        right_wing_cutout = right_wing_cutout.translate(wing_config.nose_pnt).display(name=f"{self.identifier}.cutout",
                                                                                      severity=logging.DEBUG)

        return {self.identifier: right_wing, f"{self.identifier}.cutout": right_wing_cutout}


### VaseModeCreator Test

In [34]:
vm = VaseModeRibCutoutCreator(creator_id="wing",
                 wing_index="main_wing",
                 printer_wall_thickness=printer_wall_thickness,
                 spare_support_geometry_is_round=spare_support_geometry_is_round,
                 spare_support_dimension_width=spare_support_dimension_width,
                 spare_support_dimension_height=spare_support_dimension_height,
                 leading_edge_offset=leading_edge_offset,
                 trailing_edge_offset=trailing_edge_offset,
                 minimum_rib_angle=minimum_rib_angle,
                 wing_config=wing_configuration,
                 taper_cutout=0,
                 invert_cutout=False)


In [35]:
set_defaults(
    axes=True,
    axes0=True,
    grid=[True, False, False],
    up='Z',
    control='orbit',
    
    position = (250+190,200,500),
    target = (250+190,200,0),
    zoom=0.7,
)

In [36]:
wing = vm._create_shape([],[])
wing['wing.cutout']#.combine()

seg: 1


In [37]:
_wlc_wing: Workplane = wlc._create_shape([],[])['wing']
_wlc_wing#.combine()

In [38]:
_wlc_of = wlc_of._create_shape([],[])['wing.offset']

In [39]:
my_wing = _wlc_wing.cut(_wlc_of).add(wing['wing.cutout']).combine()

In [32]:
my_wing

In [33]:
from cadquery import exporters
exporters.export(my_wing, "./wing.stl", tolerance=0.01, angularTolerance=0.1)
